In [1]:
# --- Colab: Python 3.10 + DAAM + Matplotlib Agg ---

import os
import subprocess

venv_dir = "py310_env"

# 1. Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev python3-pip git

# 2. Create virtual environment if it doesn't exist
if not os.path.exists(venv_dir):
    subprocess.run(["python3.10", "-m", "venv", venv_dir])

# 3. Upgrade pip inside venv
subprocess.run(f"source {venv_dir}/bin/activate && pip install --upgrade pip",
               shell=True, executable="/bin/bash")

# 4. Install DAAM and dependencies
subprocess.run(f"""
source {venv_dir}/bin/activate && \
pip install git+https://github.com/deepaallan/daam.git && \
pip install torch torchvision numpy matplotlib
""", shell=True, executable="/bin/bash")

# 5. Force compatible huggingface_hub version (needed for DAAM)
subprocess.run(f"""
source {venv_dir}/bin/activate && \
pip install 'huggingface_hub<1.0.0,>=0.16.4'
""", shell=True, executable="/bin/bash")

# 6. Helper function to run Python 3.10 code in venv
def py310(code):
    """Run Python 3.10 code in the virtual environment and return output"""
    # Force Matplotlib Agg backend automatically
    prepended_code = 'import os; os.environ[\"MPLBACKEND\"]=\"Agg\"; ' + code
    command = f'source {venv_dir}/bin/activate && python -c' + prepended_code
    result = subprocess.run(command, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,519 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/p

In [2]:

# 7. Test DAAM import + Matplotlib
py310("""
import daam
import matplotlib.pyplot as plt

""")



  File "<string>", line 1
    import
          ^
SyntaxError: invalid syntax
/bin/bash: line 1: os.environ[MPLBACKEND]=Agg: command not found
/bin/bash: line 2: import: command not found
/bin/bash: line 3: import: command not found



In [3]:
print('import os; os.environ[\"MPLBACKEND\"]=\"Agg\"; ')

import os; os.environ["MPLBACKEND"]="Agg"; 


In [4]:
# --- Colab: Python 3.10 + DAAM (no escaping issues) ---

import os
import subprocess
from google.colab import userdata

venv_dir = "py310_env"

# 1. Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev python3-pip git

# 2. Create virtual environment if it doesn't exist
if not os.path.exists(venv_dir):
    subprocess.run(["python3.10", "-m", "venv", venv_dir])

# 3. Upgrade pip inside venv
subprocess.run(f"source {venv_dir}/bin/activate && pip install --upgrade pip",
               shell=True, executable="/bin/bash")

# 3. Upgrade pip inside venv
subprocess.run(f"source {venv_dir}/bin/activate && pip install torch torchvision numpy matplotlib spacy scipy",
               shell=True, executable="/bin/bash")

subprocess.run(f"source {venv_dir}/bin/activate && pip install transformers==4.41.2 diffusers==0.28.0 accelerate==0.31.0 safetensors==0.4.5",
               shell=True, executable="/bin/bash")

# 5. Force compatible huggingface_hub version
subprocess.run(f"""
source {venv_dir}/bin/activate && \
pip install 'huggingface_hub==0.25.2'
""", shell=True, executable="/bin/bash")

# 4. Install DAAM
subprocess.run(f"""
source {venv_dir}/bin/activate && \
pip install --no-deps daam
""", shell=True, executable="/bin/bash") # used to say --no-deps


# 6. Helper function: write code to a temp file and run it
def py310_file(code, filename="temp_script.py"):
    """Run Python 3.10 code by writing to a temp file in the venv"""
    with open(filename, "w") as f:
        f.write(code)
    command = f"source {venv_dir}/bin/activate && python {filename}"
    result = subprocess.run(command, shell=True, executable="/bin/bash", capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    os.remove(filename)  # optional cleanup

# 7. Test DAAM import
py310_file(f"""
import os
os.environ[\"MPLBACKEND\"]=\"Agg\"
import daam
print("DAAM imported successfully!")

from daam import trace, set_seed
from diffusers import DiffusionPipeline
from matplotlib import pyplot as plt
import torch

from huggingface_hub.hf_api import HfFolder
HfFolder.save_token('{userdata.get('hftoken')}')


model_id = 'stabilityai/stable-diffusion-xl-base-1.0'
device = 'cuda'

pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True, torch_dtype=torch.float16, use_safetensors=True, variant='fp16')
pipe = pipe.to(device)

prompt = 'A criminal'
gen = set_seed(0)  # for reproducibility

with torch.no_grad():
    with trace(pipe) as tc:
        out = pipe(prompt, num_inference_steps=50, generator=gen)
        heat_map_global = tc.compute_global_heat_map()
        for word in prompt.split(' '):
          heat_map = heat_map_global.compute_word_heat_map(word)
          heat_map.plot_overlay(out.images[0])
          plt.savefig(word + '.png')


""")


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [5]:
def generate_maps(prompt):
  # 7. Test DAAM import
  py310_file(f"""
import os
os.environ[\"MPLBACKEND\"]=\"Agg\"
import daam
print("DAAM imported successfully!")

from daam import trace, set_seed
from diffusers import DiffusionPipeline
from matplotlib import pyplot as plt
import torch

from huggingface_hub.hf_api import HfFolder
HfFolder.save_token('{userdata.get('hftoken')}')


model_id = 'stabilityai/stable-diffusion-xl-base-1.0'
device = 'cuda'

pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True, torch_dtype=torch.float16, use_safetensors=True, variant='fp16')
pipe = pipe.to(device)

prompt = '{prompt}'
gen = set_seed(0)  # for reproducibility

for i in range(5):
  with torch.no_grad():
      with trace(pipe) as tc:
          out = pipe(prompt, num_inference_steps=50, generator=gen)
          heat_map_global = tc.compute_global_heat_map()
          for word in prompt.split(' '):
            heat_map = heat_map_global.compute_word_heat_map(word)
            heat_map.plot_overlay(out.images[0])
            plt.savefig(word + '_' + str(i) + '_' + prompt + '.png')


  """)

  for word in prompt.split(" "):
    # Get saved image of word.png
    image_path = f"{word}.png"


In [6]:
generate_maps('criminal')

KeyboardInterrupt: 

In [ ]:
 # Process saved images in regular python/can output to flask

In [ ]:
### Influence functions? (experimental only)

py310_file("""
import os
os.environ[\"MPLBACKEND\"]=\"Agg\"
import torch
import daam
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

# ---- custom hooks to record internal features ----
feature_maps = {}

def save_activation(name):
    def hook(model, input, output):
        # If output is a tuple, take its first tensor
        out = output[0] if isinstance(output, tuple) else output
        feature_maps[name] = out.detach().cpu()
    return hook


# Attach hooks to chosen layers of the U-Net (e.g., midblock or downblocks)
handles = []
for name, module in pipe.unet.named_modules():
    if "mid_block.resnets.0" in name or "down_blocks.2.attentions.1" in name:
        handles.append(module.register_forward_hook(save_activation(name)))

# ---- run diffusion with DAAM ----
prompt = "a small cat sitting on a wooden chair"
with daam.trace(pipe) as tracer:
    image = pipe(prompt, num_inference_steps=20).images[0]

# remove hooks
for h in handles:
    h.remove()


import torch
import numpy as np
from torchvision.transforms.functional import resize

token = "cat"
token_map = tracer.compute_global_heat_map(token)  # shape (H_img, W_img)

token_map_data = token_map.data if hasattr(token_map, "data") else token_map
token_map_array = torch.tensor(token_map_data) if not isinstance(token_map_data, torch.Tensor) else token_map_data

for layer, feat in feature_maps.items():
    # Convert token_map to a tensor or array before using it

    # Now resize using its spatial shape
    f = resize(feat[0, 0:1], token_map_array.shape, antialias=True)

    corr = torch.corrcoef(torch.stack([
        f.flatten(),
        token_map_array.flatten()
    ]))[0, 1]

    print(f"{layer}: correlation with '{token}' = {corr.item():.3f}")


import matplotlib.pyplot as plt
from torchvision.utils import make_grid

layer = list(feature_maps.keys())[0]
fm = feature_maps[layer][0][:8]  # take 8 channels

grid = make_grid(fm, nrow=4, normalize=True, scale_each=True)
plt.figure(figsize=(8,8))
plt.imshow(grid.permute(1,2,0))
plt.title(f"Internal concepts in {layer}")
plt.axis("off")
plt.savefig("test2.png")

token = "chair"
heatmap = tracer.compute_heat_map(token)
daam.overlay_heat_map(heatmap, image).savefig("test3.png")


""")